# 03 Transformation Test

This notebook is used to prototype the transformation logic before converting it into PostgreSQL stored procedures.

For each table, the structure is:

```text
1. Query the raw table
2. Apply the transformation SELECT statement

```

In order to compare the raw output with the transformed output


## 1. Connect to PostgreSQL

This notebook reuses the database connection from `src/db_connection.py`.

In [9]:
from pathlib import Path
import sys
import pandas as pd

# This assumes the notebook is inside:
# hospital_analytics_project/notebooks/
BASE_DIR = Path.cwd().parent
SRC_DIR = BASE_DIR / "src"

sys.path.append(str(SRC_DIR))

from db_connection import get_engine

engine = get_engine()

2026-06-18 12:36:45,100 | INFO | db_connection | Database environment variables validated successfully.
2026-06-18 12:36:45,100 | INFO | db_connection | Building database URL for host=localhost, port=5432, database=hospital_management, user=postgres
2026-06-18 12:36:45,100 | INFO | db_connection | Creating SQLAlchemy engine.


## 2. Helper function

This helper runs a SQL query in PostgreSQL and returns the result as a pandas DataFrame.


In [15]:
def run_query(query: str) -> pd.DataFrame:
    """Run a SQL query in PostgreSQL and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(query, conn)

# 3. Patients transformation

Business rules:
1. Standardize `gender` to `F`, `M`, or `O`.
2. Convert `date_of_birth` to `DATE`.
3. Keep only numbers in `contact_number`.
4. Convert `registration_date` to `DATE`.
5. Convert `email` to lowercase.


## 3.1 Patients — raw data

First, retrieve the data as it currently exists in the `raw.patients` table.


In [3]:
raw_patients_query = """
SELECT
    patient_id,
    first_name,
    last_name,
    gender,
    date_of_birth,
    contact_number,
    address,
    registration_date,
    insurance_provider,
    insurance_number,
    email,
    source_file
FROM raw.patients
WHERE patient_id IS NOT NULL
ORDER BY patient_id;
"""

raw_patients_df = run_query(raw_patients_query)
raw_patients_df.head(10)


,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,source_file
0,P001,David,Williams,Female,04/06/1955,(693)9585183,789 Pine Rd,23/06/2022,WellnessCorp,INS840674,david.williams@mail.com,patients_manual.csv
1,P002,Emily,Smith,Female,12/10/1984,8228188767,321 Maple Dr,15/01/2022,PulseSecure,INS354079,emily.smith@mail.com,patients_manual.csv
2,P003,Laura,Jones,MALE,21/08/1977,(839)7029847,321 Maple Dr,07/02/2022,PulseSecure,INS650929,LAURA.JONES@MAIL.COM,patients_manual.csv
3,P004,Michael,Johnson,Female,20/02/1981,(901)9443432,123 Elm St,02/03/2021,HealthIndia,INS789944,MICHAEL.JOHNSON@MAIL.COM,patients_manual.csv
4,P005,David,Wilson,Male,23/06/1960,7734463155,123 Elm St,29/09/2021,MedCare Plus,INS788105,DAVID.WILSON@MAIL.COM,patients_manual.csv
5,P006,Linda,Jones,Male,16/06/1963,7561777264,321 Maple Dr,02/10/2022,HealthIndia,INS613758,linda.jones@mail.com,patients_manual.csv
6,P007,Alex,Johnson,Female,08/06/1989,627-871-0077,789 Pine Rd,25/12/2021,MedCare Plus,INS465890,alex.johnson@mail.com,patients_manual.csv
7,P008,David,Davis,Female,05/07/1976,7090558393,456 Oak Ave,25/05/2021,WellnessCorp,INS545101,david.davis@mail.com,patients_manual.csv
8,P009,Laura,Davis,MALE,11/12/1971,(706)0324619,321 Maple Dr,18/09/2022,PulseSecure,INS136631,LAURA.DAVIS@MAIL.COM,patients_manual.csv
9,P010,Michael,Taylor,Male,13/10/2001,708-139-6733,123 Elm St,24/08/2022,WellnessCorp,INS866577,MICHAEL.TAYLOR@MAIL.COM,patients_manual.csv


## 3.2 Patients — transformed data

Now apply the transformation logic using a `SELECT` statement.


In [4]:
transformed_patients_query = """
SELECT
    patient_id,
    first_name,
    last_name,
    CASE
        WHEN LOWER(TRIM(gender)) = 'female' THEN 'F'
        WHEN LOWER(TRIM(gender)) = 'male' THEN 'M'
        WHEN LOWER(TRIM(gender)) = 'f' THEN 'F'
        WHEN LOWER(TRIM(gender)) = 'm' THEN 'M'
        ELSE 'O'
    END AS gender,
    TO_DATE(date_of_birth, 'DD/MM/YYYY') AS date_of_birth,
    REGEXP_REPLACE(contact_number, '[^0-9]', '', 'g') AS contact_number,
    address,
    TO_DATE(registration_date, 'DD/MM/YYYY') AS registration_date,
    insurance_provider,
    insurance_number,
    LOWER(TRIM(email)) AS email,
    source_file
FROM raw.patients
WHERE patient_id IS NOT NULL
ORDER BY patient_id;
"""

patients_df = run_query(transformed_patients_query)
patients_df.head(10)


,patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,source_file
0,P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,patients_manual.csv
1,P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,patients_manual.csv
2,P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,patients_manual.csv
3,P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,patients_manual.csv
4,P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com,patients_manual.csv
5,P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com,patients_manual.csv
6,P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,patients_manual.csv
7,P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com,patients_manual.csv
8,P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,patients_manual.csv
9,P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com,patients_manual.csv


# 4. Doctors transformation

Business rules:
1. Format `specialization` using title case.
2. Keep only numbers in `phone_number`.
3. Convert `years_experience` to integer.
4. Convert `email` to lowercase.


## 4.1 Doctors — raw data

In [ ]:
raw_doctors_query = """
SELECT
    doctor_id,
    first_name,
    last_name,
    specialization,
    phone_number,
    years_experience,
    hospital_branch,
    email,
    source_file
FROM raw.doctors
WHERE doctor_id IS NOT NULL
ORDER BY doctor_id;
"""

raw_doctors_df = run_query(raw_doctors_query)
raw_doctors_df.head(10)


## 4.2 Doctors — transformed data

In [6]:
transformed_doctors_query = """
SELECT
    doctor_id,
    first_name,
    last_name,
    INITCAP(TRIM(specialization)) AS specialization,
    REGEXP_REPLACE(phone_number, '[^0-9]', '', 'g') AS phone_number,
    NULLIF(years_experience, '')::INTEGER AS years_experience,
    hospital_branch,
    LOWER(TRIM(email)) AS email,
    source_file
FROM raw.doctors
WHERE doctor_id IS NOT NULL
ORDER BY doctor_id;
"""

doctors_df = run_query(transformed_doctors_query)
doctors_df.head(10)


,doctor_id,first_name,last_name,specialization,phone_number,years_experience,hospital_branch,email,source_file
0,D001,David,Taylor,Dermatology,8322010158,17,Westside Clinic,dr.david.taylor@hospital.com,doctors01.csv
1,D002,Jane,Davis,Pediatrics,9004382050,24,Eastside Clinic,dr.jane.davis@hospital.com,doctors01.csv
2,D003,Jane,Smith,Pediatrics,8737740598,19,Eastside Clinic,dr.jane.smith@hospital.com,doctors01.csv
3,D004,David,Jones,Pediatrics,6594221991,28,Central Hospital,dr.david.jones@hospital.com,doctors01.csv
4,D005,Sarah,Taylor,Dermatology,9118538547,26,Central Hospital,dr.sarah.taylor@hospital.com,doctors01.csv
5,D006,Alex,Davis,Pediatrics,6570137231,23,Central Hospital,dr.alex.davis@hospital.com,doctors01.csv
6,D007,Robert,Davis,Oncology,8217493115,26,Westside Clinic,dr.robert.davis@hospital.com,doctors01.csv
7,D008,Linda,Brown,Dermatology,9069162601,5,Westside Clinic,dr.linda.brown@hospital.com,doctors01.csv
8,D009,Sarah,Smith,Pediatrics,7387087517,26,Central Hospital,dr.sarah.smith@hospital.com,doctors01.csv
9,D010,Linda,Wilson,Oncology,6176383634,21,Eastside Clinic,dr.linda.wilson@hospital.com,doctors01.csv


# 5. Appointments transformation

Business rules:
1. Convert `appointment_date` to `DATE`.
2. Convert `appointment_time` to `TIME`.
3. Format `status` using title case.


## 5.1 Appointments — raw data

In [7]:
raw_appointments_query = """
SELECT
    appointment_id,
    patient_id,
    doctor_id,
    appointment_date,
    appointment_time,
    reason_for_visit,
    status,
    source_file
FROM raw.appointments
WHERE appointment_id IS NOT NULL
ORDER BY appointment_id;
"""

raw_appointments_df = run_query(raw_appointments_query)
raw_appointments_df.head(10)


,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,source_file
0,A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled,appointments01.csv
1,A002,P032,D004,2023-06-09,2:30 pm,Therapy,No-show,appointments01.csv
2,A003,P048,D004,2023-06-28,8:00 AM,Consultation,Cancelled,appointments01.csv
3,A004,P025,D006,2023-09-01,9:15:00,Consultation,Cancelled,appointments01.csv
4,A005,P040,D003,2023-07-06,12:45:00,Emergency,No-show,appointments01.csv
5,A006,P045,D006,2023-06-19,4:15 pm,Checkup,Scheduled,appointments01.csv
6,A007,P001,D007,2023-04-09,10:30 am,Consultation,Scheduled,appointments01.csv
7,A008,P016,D010,2023-05-24,8:45 am,Consultation,Cancelled,appointments01.csv
8,A009,P039,D010,2023-03-05,1:45 PM,Follow-up,Scheduled,appointments01.csv
9,A010,P005,D003,2023-01-13,3:30 pm,Therapy,Completed,appointments01.csv


## 5.2 Appointments — transformed data

In [13]:
transformed_appointments_query = """
SELECT
    appointment_id,
    patient_id,
    doctor_id,
    TO_DATE(appointment_date, 'YYYY-MM-DD') AS appointment_date,
    CASE
        WHEN UPPER(appointment_time) LIKE '%%AM%%'
          OR UPPER(appointment_time) LIKE '%%PM%%'
        THEN TO_TIMESTAMP(appointment_time, 'HH12:MI AM')::TIME
        ELSE appointment_time::TIME
    END AS appointment_time,
    reason_for_visit,
    INITCAP(TRIM(status)) AS status,
    source_file
FROM raw.appointments
WHERE appointment_id IS NOT NULL
ORDER BY appointment_id;
"""

appointments_df = run_query(transformed_appointments_query)
appointments_df.head(10)


,appointment_id,patient_id,doctor_id,appointment_date,appointment_time,reason_for_visit,status,source_file
0,A001,P034,D009,2023-08-09,15:15:00,Therapy,Scheduled,appointments01.csv
1,A002,P032,D004,2023-06-09,14:30:00,Therapy,No-Show,appointments01.csv
2,A003,P048,D004,2023-06-28,08:00:00,Consultation,Cancelled,appointments01.csv
3,A004,P025,D006,2023-09-01,09:15:00,Consultation,Cancelled,appointments01.csv
4,A005,P040,D003,2023-07-06,12:45:00,Emergency,No-Show,appointments01.csv
5,A006,P045,D006,2023-06-19,16:15:00,Checkup,Scheduled,appointments01.csv
6,A007,P001,D007,2023-04-09,10:30:00,Consultation,Scheduled,appointments01.csv
7,A008,P016,D010,2023-05-24,08:45:00,Consultation,Cancelled,appointments01.csv
8,A009,P039,D010,2023-03-05,13:45:00,Follow-up,Scheduled,appointments01.csv
9,A010,P005,D003,2023-01-13,15:30:00,Therapy,Completed,appointments01.csv


# 6. Treatments transformation

Business rules:
1. Format `treatment_type` using title case.
2. Convert `treatment_date` to `DATE`.


## 6.1 Treatments — raw data

In [9]:
raw_treatments_query = """
SELECT
    treatment_id,
    appointment_id,
    treatment_type,
    description,
    treatment_date,
    source_file
FROM raw.treatments
WHERE treatment_id IS NOT NULL
ORDER BY treatment_id;
"""

raw_treatments_df = run_query(raw_treatments_query)
raw_treatments_df.head(10)


,treatment_id,appointment_id,treatment_type,description,treatment_date,source_file
0,T001,A001,Chemotherapy,Basic screening,09/08/2023,treatments01.csv
1,T002,A002,MRI,Advanced protocol,09/06/2023,treatments01.csv
2,T003,A003,MRI,Standard procedure,28/06/2023,treatments01.csv
3,T004,A004,MRI,Basic screening,01/09/2023,treatments01.csv
4,T005,A005,ECG,Standard procedure,06/07/2023,treatments01.csv
5,T006,A006,Chemotherapy,Standard procedure,19/06/2023,treatments01.csv
6,T007,A007,Chemotherapy,Advanced protocol,09/04/2023,treatments01.csv
7,T008,A008,Physiotherapy,Basic screening,24/05/2023,treatments01.csv
8,T009,A009,Physiotherapy,Standard procedure,05/03/2023,treatments01.csv
9,T010,A010,Physiotherapy,Standard procedure,13/01/2023,treatments01.csv


## 6.2 Treatments — transformed data

In [10]:
transformed_treatments_query = """
SELECT
    treatment_id,
    appointment_id,
    INITCAP(TRIM(treatment_type)) AS treatment_type,
    description,
    TO_DATE(treatment_date, 'DD/MM/YYYY') AS treatment_date,
    source_file
FROM raw.treatments
WHERE treatment_id IS NOT NULL
ORDER BY treatment_id;
"""

treatments_df = run_query(transformed_treatments_query)
treatments_df.head(10)


,treatment_id,appointment_id,treatment_type,description,treatment_date,source_file
0,T001,A001,Chemotherapy,Basic screening,2023-08-09,treatments01.csv
1,T002,A002,Mri,Advanced protocol,2023-06-09,treatments01.csv
2,T003,A003,Mri,Standard procedure,2023-06-28,treatments01.csv
3,T004,A004,Mri,Basic screening,2023-09-01,treatments01.csv
4,T005,A005,Ecg,Standard procedure,2023-07-06,treatments01.csv
5,T006,A006,Chemotherapy,Standard procedure,2023-06-19,treatments01.csv
6,T007,A007,Chemotherapy,Advanced protocol,2023-04-09,treatments01.csv
7,T008,A008,Physiotherapy,Basic screening,2023-05-24,treatments01.csv
8,T009,A009,Physiotherapy,Standard procedure,2023-03-05,treatments01.csv
9,T010,A010,Physiotherapy,Standard procedure,2023-01-13,treatments01.csv


# 7. Billing transformation

Business rules:
1. Convert `bill_date` to `DATE`.
2. Clean `amount` and convert it to `NUMERIC`.
3. Standardize `payment_method`.
4. Format `payment_status` using title case.


## 7.1 Billing — raw data

In [11]:
raw_billing_query = """
SELECT
    bill_id,
    patient_id,
    treatment_id,
    bill_date,
    amount,
    payment_method,
    payment_status,
    source_file
FROM raw.billing
WHERE bill_id IS NOT NULL
ORDER BY bill_id;
"""

raw_billing_df = run_query(raw_billing_query)
raw_billing_df.head(10)


,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,source_file
0,B001,P034,T001,09/08/2023,3941.97,Insurance,Pending,billing01.csv
1,B002,P032,T002,09/06/2023,"$4,158.44",INSURANCE,Paid,billing01.csv
2,B003,P048,T003,28/06/2023,3731.55,insurance,Paid,billing01.csv
3,B004,P025,T004,01/09/2023,4799.86,Insurance,Failed,billing01.csv
4,B005,P040,T005,06/07/2023,582.05,Credit Card,Pending,billing01.csv
5,B006,P045,T006,19/06/2023,"1,381.00",Insurance,Pending,billing01.csv
6,B007,P001,T007,09/04/2023,534.03,CASH,Failed,billing01.csv
7,B008,P016,T008,24/05/2023,3413.64,cash,Failed,billing01.csv
8,B009,P039,T009,05/03/2023,"4,541.14",credit card,Paid,billing01.csv
9,B010,P005,T010,13/01/2023,1595.67,CASH,Paid,billing01.csv


## 7.2 Billing — transformed data

In [12]:
transformed_billing_query = """
SELECT
    bill_id,
    patient_id,
    treatment_id,
    TO_DATE(bill_date, 'DD/MM/YYYY') AS bill_date,
    NULLIF(REGEXP_REPLACE(amount, '[^0-9.]', '', 'g'), '')::NUMERIC(10,2) AS amount,
    CASE
        WHEN LOWER(REPLACE(payment_method, ' ', '')) = 'insurance'
            THEN 'Insurance'
        WHEN LOWER(REPLACE(payment_method, ' ', '')) = 'creditcard'
            THEN 'Credit Card'
        WHEN LOWER(REPLACE(payment_method, ' ', '')) = 'cash'
            THEN 'Cash'
        ELSE 'Unknown'
    END AS payment_method,
    INITCAP(TRIM(payment_status)) AS payment_status,
    source_file
FROM raw.billing
WHERE bill_id IS NOT NULL
ORDER BY bill_id;
"""

billing_df = run_query(transformed_billing_query)
billing_df.head(10)


,bill_id,patient_id,treatment_id,bill_date,amount,payment_method,payment_status,source_file
0,B001,P034,T001,2023-08-09,3941.97,Insurance,Pending,billing01.csv
1,B002,P032,T002,2023-06-09,4158.44,Insurance,Paid,billing01.csv
2,B003,P048,T003,2023-06-28,3731.55,Insurance,Paid,billing01.csv
3,B004,P025,T004,2023-09-01,4799.86,Insurance,Failed,billing01.csv
4,B005,P040,T005,2023-07-06,582.05,Credit Card,Pending,billing01.csv
5,B006,P045,T006,2023-06-19,1381.00,Insurance,Pending,billing01.csv
6,B007,P001,T007,2023-04-09,534.03,Cash,Failed,billing01.csv
7,B008,P016,T008,2023-05-24,3413.64,Cash,Failed,billing01.csv
8,B009,P039,T009,2023-03-05,4541.14,Credit Card,Paid,billing01.csv
9,B010,P005,T010,2023-01-13,1595.67,Cash,Paid,billing01.csv
